In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import random
from uuid import uuid4
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import getopt
import sys
import os
import math
import time
import argparse
from visdom import Visdom
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

sys.path.insert(0, os.path.join('..', '..'))

import torch as T
from torch.autograd import Variable as var
import torch.nn.functional as F
import torch.optim as optim

from torch.nn.utils import clip_grad_norm_

from dnc.dnc import DNC
from dnc.sdnc import SDNC
from dnc.sam import SAM
from dnc.util import *

from dnc.lib import exp_loss, InputStorage, mse, criterion, ENDSYM, tensor2string

In [2]:
viz = Visdom()
# assert viz.check_connection()


def llprint(message):
  sys.stdout.write(message)
  sys.stdout.flush()



st = InputStorage()


def generate_data(batch_size, length, maxlength, testoccurance=True, transposeInput=False, transposeOutput=False):
  input_data = np.zeros((batch_size, maxlength, maxlength), dtype=np.float32)
  target_output = np.zeros((batch_size, maxlength, maxlength), dtype=np.float32)
  sequence1 = np.random.binomial(1, 0.5, (batch_size, length, 1))
  sequence2 = np.random.binomial(1, 0.5, (batch_size, length, 1))

  if testoccurance: # test if the sequence is in the test data, replace if so
    for i in range(batch_size):
      input_test_data = np.zeros((1, maxlength, maxlength), dtype=np.float32)
      input_test_data[0, 0:length, 0:1] = sequence1[i] #first sequence
      input_test_data[0, length, 1] = ENDSYM  #pause
      input_test_data[0, length+1:length*2+1, 2:3] = sequence2[i] #second sequence
      input_test_data[0, length*2+1, 3] = ENDSYM  #pause
      while st.isSaved(input_test_data[0], flag="testData"):
        if np.random.binomial(1, 0.5, 1) == 1: # replace first sequence
          sequence1[i] = np.random.binomial(1, 0.5, (length, 1))
          input_test_data[0, 0:length, 0:1] = sequence1[i]
        else: # replace second sequence
          sequence2[i] = np.random.binomial(1, 0.5, (length, 1))
          input_test_data[0, length+1:length*2+1, 2:3] = sequence2[i]

  input_data[:, 0:length, 0:1] = sequence1 #first sequence
  input_data[:, length, 1] = ENDSYM  #pause
  input_data[:, length+1:length*2+1, 2:3] = sequence2 #second sequence
  input_data[:, length*2+1, 3] = ENDSYM  #pause
  if transposeInput:
    for i in range(batch_size):
      input_data[i] = input_data[i].T

  def calcsum(sequenceA, sequenceB): #calculate sum of two binary numbers
    sumsequence = np.zeros((batch_size, length + 1, length +1))
    assert len(sequenceA) == len(sequenceB)
    for k in range(len(sequenceA)):
      carry = 0 # carry bit
      for j in reversed(range(len(sequenceA[k]))):
            if sequenceA[k][j][0] == 1 and sequenceB[k][j][0] == 1: #1+1=10
                sumsequence[k][j+1][-1] = 0+carry
                carry = 1
            elif (sequenceA[k][j][0] == 1 and sequenceB[k][j][0] == 0) or (sequenceA[k][j][0] == 0 and sequenceB[k][j][0] == 1): #1+0=1 and 0+1=1
                if carry == 1:
                    sumsequence[k][j+1][-1] = 0
                    carry = 1
                else:
                    sumsequence[k][j+1][-1] = 1
                    carry = 0
            else:
                sumsequence[k][j+1][-1] = 0+carry #0+0=0
                carry = 0
      sumsequence[k][0][-1] = carry
    return sumsequence
  
  cs = calcsum(sequence1, sequence2)
  for i in range(batch_size):
    target_output[i, -(length+1):, -(length+1):] = cs[i] #write sum to target output
    if transposeOutput:
      target_output[i] = target_output[i].T

  return input_data, target_output






def incrementCurriculum(trainError, epoch, sequence_length, maxsequence_length, curriculum_fre):
  return epoch != 0 and sequence_length < maxsequence_length and epoch % curriculum_fre == 0

Setting up a new session...


In [3]:
import copy
from dnc.lib import STEPBYSTEPOBJ
import pickle

import os

batch_size = 100
sequence_length = 3
sequence_max_length = 6 
iterations = int(2.5*10**3) #200000
summarize_freq = int(iterations // 25)
check_freq = int(iterations // 25)
curriculum_freq = 750


  # input_size = output_size = args.input_size
mem_slot = 32
mem_size = 1
read_heads = 1
curriculum_increment = 1
input_size = 2*sequence_max_length + 2
output_size = 64

replaceWithWrong = True

In [4]:
lossfn = mse

comp_obj = []

for rh_i in [1,2,3,5]:
  rnn = DNC(
        input_size=input_size,
        hidden_size=output_size,
        rnn_type='rnn',
        #rnn_type='lstm',
        num_layers=rh_i,
        num_hidden_layers=1,
        dropout=0,
        nr_cells=mem_slot,
        cell_size=mem_size,
        read_heads=read_heads,
        gpu_id=-1,
        debug='store_true',
        batch_first=True,
        independent_linears=True,
        nonlinearity='tanh',
    )
  optimizer = optim.Adam(rnn.parameters(), lr=0.001, eps=1e-9, betas=[0.9, 0.98]) # 0.0001
  comp_obj.append({"lossfn": lossfn, "rnn": rnn, "optimizer": optimizer, "chx": None, "mhx": None, "rv": None, "last_save_losses": [], "datas": [], "i": rh_i})

for cobj in comp_obj:
  print(cobj["lossfn"].__name__)

def create_directory_if_not_exists(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print("Directory created successfully!")
    else:
        print("Directory already exists.")

name = 'add_' + str(uuid4().hex)[:3] + '_comp_layers'

lastcp = None

create_directory_if_not_exists(name)





loadcp = False #= 'checkpoint_add_46_242000.pth

print(input_size, output_size)


with open(f'{name}/output.txt', 'w') as f:
  print(name)
  print(name, file=f)
  
  

  last_save_losses = []

  for i in range(3, sequence_max_length,1): # generate test data
    inputdataspace = 2**i*2 # 2 i bit sequences
    testdatasize = int(inputdataspace*0.15)+1 #15%
    input_data, target_output = generate_data(testdatasize, i, input_size)
    for i in range(testdatasize):
      st.saveInput(input_data[i], output=target_output[i], withoutIncrement=True, flag="testData") #saveData


  
  Testloss = 0 # loss of test data
  for epoch in tqdm(range(iterations + 1)):
    #llprint("\rIteration {ep}/{tot}".format(ep=epoch, tot=iterations))
    for comp in comp_obj:
      optimizer = comp["optimizer"]
      rnn = comp["rnn"]
      chx = comp["chx"]
      mhx = comp["mhx"]
      rv = comp["rv"]
      last_save_losses = comp["last_save_losses"]
      datas = comp["datas"]
      combLoss = comp["lossfn"]
      lossfni = comp["i"]
      optimizer.zero_grad()


      input_data, target_output = generate_data(batch_size, sequence_length, input_size) # generate data
      input_data = var(T.from_numpy(input_data))
      target_output = var(T.from_numpy(target_output))


      if rnn.debug:
        output, (chx, mhx, rv), v = rnn(input_data, (None, mhx, None), reset_experience=True, pass_through_memory=True)
      else:
        output, (chx, mhx, rv) = rnn(input_data, (None, mhx, None), reset_experience=True, pass_through_memory=True)

      loss = combLoss((output), target_output)
      

      if epoch % 100 == 0:
        testset = st.getDataByFlag("testData") # get test data
        testlosses = []
        for k in range(int(len(testset) / batch_size)+1): # split testdata into batch_size chunks
          input_TEST_data = np.zeros((batch_size, input_size, input_size), dtype=np.float32)
          target_TEST_output = np.zeros((batch_size, input_size, input_size), dtype=np.float32)
          for i in range(batch_size):
            if i + k * batch_size < len(testset):
              input_TEST_data[i] = testset[k*batch_size+i]["input"]
              target_TEST_output[i] = testset[k*batch_size+i]["output"]
            else: # if there is not enough test data fill the remaining slots with random entries
              robj = random.choice(testset)
              input_TEST_data[i] = robj["input"]
              target_TEST_output[i] = robj["output"]

          input_TEST_data = var(T.from_numpy(input_TEST_data))
          target_TEST_output = var(T.from_numpy(target_TEST_output))
          if rnn.debug:
            TEST_output, _, _ = rnn(input_data, (None, mhx, None), reset_experience=True, pass_through_memory=True)
          else:
            TEST_output, _ = rnn(input_data, (None, mhx, None), reset_experience=True, pass_through_memory=True)

          ri = random.randint(0, batch_size-1)
          print("TEST_input: \n", tensor2string(input_TEST_data[ri]), file=f)
          print("TEST_output: \n", tensor2string(TEST_output[ri]), file=f)
          print("target_TEST_output: \n", tensor2string(target_TEST_output[ri]), file=f)
          print("CE Loss: ", str(mse(TEST_output[ri], target_TEST_output[ri]).item()), file=f)

          MyTestloss = combLoss((TEST_output), target_TEST_output).item() # calculate test loss
          testlosses.append(MyTestloss)
        Testloss = np.mean(testlosses) # calculate test loss mean

      datas.append({"epoch": epoch, "loss": loss.item(), "testloss": Testloss, "sequencelength": sequence_length, "layers": lossfni}) #append to the datas df
      
      loss.backward()
      T.nn.utils.clip_grad_norm_(rnn.parameters(), 50)
      optimizer.step()
      loss_value = loss.item()

      summarize = (epoch % summarize_freq == 0)
      take_checkpoint = (epoch != 0) and (epoch % check_freq == 0)
      

      # detach memory from graph
      mhx = { k : (v.detach() if isinstance(v, var) else v) for k, v in mhx.items() }

      last_save_losses.append(loss_value)
      loss = np.mean(last_save_losses)

      if summarize:
        llprint("\n\tAvg. Loss: %.4f\n" % (loss))
        llprint("\n\tAvg. Test Loss: %.4f\n" % (Testloss))
        if np.isnan(loss):
          raise Exception('nan Loss')
        print("\n")

      if summarize and rnn.debug:
        last_save_losses = []

        viz.heatmap(
              v['memory'],
              opts=dict(
                  xtickstep=10,
                  ytickstep=2,
                  title= name + 'Memory, t: ' + str(epoch) + ', loss: ' + str(loss),
                  ylabel='layer * time',
                  xlabel='mem_slot * mem_size'
              )
          )

        viz.heatmap(
              v['link_matrix'][-1].reshape(mem_slot, mem_slot),
              opts=dict(
                  xtickstep=10,
                  ytickstep=2,
                  title=name + 'Link Matrix, t: ' + str(epoch) + ', loss: ' + str(loss),
                  ylabel='mem_slot',
                  xlabel='mem_slot'
              )
        )
      
        viz.heatmap(
              v['precedence'],
              opts=dict(
                  xtickstep=10,
                  ytickstep=2,
                  title=name + 'Precedence, t: ' + str(epoch) + ', loss: ' + str(loss),
                  ylabel='layer * time',
                  xlabel='mem_slot'
              )
        )

      if incrementCurriculum(loss, epoch, sequence_length, sequence_max_length, curriculum_freq):
        sequence_length = sequence_length + curriculum_increment
        print("Increasing max length to " + str(sequence_length))

      comp["chx"] = chx
      comp["mhx"] = mhx
      comp["rv"] = rv
      comp["last_save_losses"] = last_save_losses
      comp["datas"] = datas
      comp["rnn"] = rnn

      if take_checkpoint:
        cur_weights = rnn.state_dict()
        T.save(cur_weights, f'{name}/checkpoint_lay{lossfni}_{epoch}.pth')
        lastcp = f'{name}/checkpoint_lay{lossfni}_{epoch}.pth'
        df = pd.DataFrame(datas)
        pickle.dump(df, open(f"{name}/df_lay{lossfni}_{epoch}.pkl", "wb"))

  
    




mse
mse
mse
mse
Directory created successfully!
14 64
add_4e8_comp_layers


  0%|          | 0/2501 [00:00<?, ?it/s]


	Avg. Loss: 0.0751

	Avg. Test Loss: 0.0816



	Avg. Loss: 0.0427

	Avg. Test Loss: 0.0497



	Avg. Loss: 0.0595

	Avg. Test Loss: 0.0679



	Avg. Loss: 0.0502

	Avg. Test Loss: 0.0525


  0%|          | 1/2501 [00:15<10:25:21, 15.01s/it]

  4%|▍         | 100/2501 [03:11<1:03:11,  1.58s/it]

...

 96%|█████████▌| 2401/2501 [1:11:29<04:50,  2.91s/it]

100%|█████████▉| 2500/2501 [1:14:58<00:02,  2.14s/it]


	Avg. Loss: 0.0069

	Avg. Test Loss: 0.0116



	Avg. Loss: 0.0022

	Avg. Test Loss: 0.0161



	Avg. Loss: 0.0022

	Avg. Test Loss: 0.0152



	Avg. Loss: 0.0021

	Avg. Test Loss: 0.0162


100%|██████████| 2501/2501 [1:15:03<00:00,  1.80s/it]

Exception: Sequence lengths do not match

In [6]:
cdatas = [{"epoch": None, "sequencelength": None } for _ in range(len(comp_obj[0]["datas"]))]
  
for comp in comp_obj:
    lossfni = comp["i"]
    datas = comp["datas"]
    df = pd.DataFrame(datas) # plot loss 
    pickle.dump(df, open(f"{name}/df_lay{lossfni}_total.pkl", "wb"))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["epoch"], y=df["loss"], mode='lines', name='Train Data'))
    #if comp["lossfn"].__name__ != mse.__name__:
    #  fig.add_trace(go.Scatter(x=df["epoch"], y=df["lossmse"], mode='lines', name='MSE Loss (train Data)'))
    #if comp["lossfn"].__name__ != criterion.__name__:
    #  fig.add_trace(go.Scatter(x=df["epoch"], y=df["losscriterion"], mode='lines', name='CrossEntropy Loss (train Data)'))
    #if comp["lossfn"].__name__ != exp_loss.__name__:
    #  fig.add_trace(go.Scatter(x=df["epoch"], y=df["lossexp"], mode='lines', name='Exp Loss (train Data)'))
    fig.add_trace(go.Scatter(x=df["epoch"], y=df["testloss"], mode='lines', name='Test Data'))
    fig.update_layout(title=f'Losses trained with {lossfni}-layers', xaxis_title='Epoch', yaxis_title='Loss')
    fig.show()
    fig.write_html(f"{name}/losses_lay{lossfni}.html")
    fig.write_image(f"{name}/losses_lay{lossfni}.png")

    for index, el in enumerate(datas):
      if cdatas[index]["epoch"] is not None and cdatas[index]["epoch"] != el["epoch"]:
        raise Exception("Epochs do not match")
      #if cdatas[index]["sequencelength"] is not None and cdatas[index]["sequencelength"] != el["sequencelength"]:
      #  raise Exception("Sequence lengths do not match")
      if cdatas[index]["epoch"] is None and cdatas[index]["epoch"] != el["epoch"]:
        cdatas[index]["epoch"] = el["epoch"]
        cdatas[index]["sequencelength"] = el["sequencelength"]

      cdatas[index][f"lay{lossfni}.loss"] = el["loss"]
      cdatas[index][f"lay{lossfni}.testloss"] = el["testloss"]

df = pd.DataFrame(cdatas)
pickle.dump(df, open(f"{name}/df_lay_alle_total.pkl", "wb"))
fig = go.Figure()
for obj in (comp_obj):
  i = obj["i"]
  fig.add_trace(go.Scatter(x=df["epoch"], y=df[f"lay{i}.loss"], mode='lines', name=f'Train Data {i} layers'))
  fig.add_trace(go.Scatter(x=df["epoch"], y=df[f"lay{i}.testloss"], mode='lines', name=f'Test Data {i} layers'))
fig.update_layout(title='Losses trained with different numbers of layers', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()
fig.write_html(f"{name}/losses_lay_alle.html")
fig.write_image(f"{name}/losses_lay_alle.png")

In [5]:
del rnn

rnn = DNC(
        input_size=input_size,
        hidden_size=output_size,
        rnn_type='rnn',
        #rnn_type='lstm',
        num_layers=2,
        num_hidden_layers=1,
        dropout=0,
        nr_cells=mem_slot,
        cell_size=mem_size,
        read_heads=read_heads,
        gpu_id=-1,
        debug='store_true',
        batch_first=True,
        independent_linears=True,
        nonlinearity='tanh',
    )

#name = 'add_a2b'
#lastcp = f'{name}/checkpoint_1000.pth'
print(name)

with open(f"{name}/output_2.txt", "w") as f:
  batch_size=1
  rnn.load_state_dict(T.load(lastcp, weights_only=True))
  rnn.eval()
  
  stepByStep = copy.deepcopy(STEPBYSTEPOBJ)

  i=0
  llprint("\nIteration %d/%d" % (i, iterations))
  # We test now the learned generalization using sequence_max_length examples
  random_length = np.random.randint(2, sequence_length  + 1)
  input_data, target_output = generate_data(1, random_length, input_size)

  #print (input_data, target_output)

  
  input_data = var(T.from_numpy(input_data))
  target_output = var(T.from_numpy(target_output))

  stepByStep["CurrI"] = i
  stepByStep["currentObj"] = copy.deepcopy(stepByStep["defObj"])
  stepByStep["currentObj"]["i"] = i 
  stepByStep["input"] = input_data.detach().numpy()
  stepByStep["target"] = target_output.detach().numpy()
  stepByStep["MEMORYCOLUMNS"] = mem_slot
  stepByStep["INPUTSIZE"] = input_size
  stepByStep["OUTPUTSIZE"] = output_size
    
  if rnn.debug:
    output, (chx, mhx, rv), v = rnn(input_data, (None, None, None), reset_experience=True, pass_through_memory=True, stepByStep=stepByStep)
  else:
    output, (chx, mhx, rv) = rnn(input_data, (None, None, None), reset_experience=True, pass_through_memory=True, stepByStep=stepByStep)

  stepByStep["output"] = output.detach().numpy()
  stepByStep["objects"].append(copy.deepcopy(stepByStep["currentObj"]))
  stepByStep['loss'] = str(mse(output, target_output).item())
  #output = output[:, -1, :].sum().data.cpu().numpy()
  #target_output = target_output.sum().data.cpu().numpy()

  print(stepByStep["input"].shape)
  print(stepByStep["output"].shape)
  print(stepByStep["target"].shape)
  #raise Exception("STOP")

  print(stepByStep)

  pickle.dump(stepByStep, open(f"{name}/stepByStep.pkl", "wb"))

  print("\n\n")
  print("Input: ", tensor2string(input_data[0]), file=f)
  print("Output: ", tensor2string(torch.round(output[0], decimals=1)), file=f)
  print("Target: ", tensor2string(target_output[0]), file=f)
  print("CE Loss: ", str(mse(output[0], target_output[0]).item()), file=f)
  print("Log Loss: ", str(criterion(output[0], target_output[0]).item()), file=f)
  print("Exp Loss: ", str(exp_loss(output[0], target_output[0]).item()), file=f)
  print("\n\n")
  print("CE Loss: ", str(mse(output, target_output).item()), file=f)
  print("Log Loss: ", str(criterion(output, target_output).item()), file=f)
  print("Exp Loss: ", str(exp_loss(output, target_output).item()), file=f)
  print("\n\n")

  try:
    print("\nReal value: ", ' = ' + str(int(target_output[0])))
    print("Predicted:  ", ' = ' + str(int(output // 1)) + " [" + str(output) + "]")
  except Exception as e:
    pass

  

NameError: name 'read_heads' is not defined